# Evaluate Tree-RL checkpoints epoch 1-25

This notebook evaluates every checkpoint in `models/epoch_001.pth` through `models/epoch_025.pth` without editing the training notebook or model files.

It reports VOC 2007 test recall for each model and saves the results to `tree_rl_all_epochs_eval.csv`.


In [1]:
# ============================================================
# Cell 1: User settings
# ============================================================
from pathlib import Path

PROJECT_DIR = Path.cwd()
MODEL_DIR = PROJECT_DIR / 'models'
FEATURE_CACHE_DIR = PROJECT_DIR / 'feature_cache_eval'
RESULTS_CSV = PROJECT_DIR / 'tree_rl_all_epochs_eval.csv'

# Use None to evaluate the full VOC 2007 test split. Set a small number
# like 100 for a quick smoke test.
MAX_IMAGES = None

# Paper table uses levels=5, which produces 31 proposals.
LEVELS = 5
IOU_THRESHOLDS = [0.5, 0.6, 0.7, 0.8]
SIZE_THRESHOLD = 2000

# If VOC is missing, the download cell can fetch VOC 2007 test automatically.
VOC_ROOT_CANDIDATES = [
    PROJECT_DIR / 'VOCdevkit' / 'VOCdevkit',
    PROJECT_DIR / 'VOCdevkit',
    Path('/content/drive/MyDrive/tree_rl_project/VOCdevkit/VOCdevkit'),
    Path('/content/drive/MyDrive/tree_rl_project/VOCdevkit'),
]

print('Project:', PROJECT_DIR)
print('Models:', MODEL_DIR)
print('Feature cache:', FEATURE_CACHE_DIR)


Project: /home/hm/4th_sem/study/cs106/treerl
Models: /home/hm/4th_sem/study/cs106/treerl/models
Feature cache: /home/hm/4th_sem/study/cs106/treerl/feature_cache_eval


In [2]:
# ============================================================
# Cell 2: Imports and device
# ============================================================
import csv
import os
import random
import tarfile
import urllib.request
import xml.etree.ElementTree as ET
from collections import deque
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.ops as ops
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
# ============================================================
# Cell 3: Optional VOC 2007 test download
# ============================================================
def find_voc_root() -> Optional[Path]:
    for root in VOC_ROOT_CANDIDATES:
        if (root / 'VOC2007' / 'ImageSets' / 'Main' / 'test.txt').exists():
            return root
    return None

def download_voc2007_test():
    base = PROJECT_DIR / 'VOCdevkit'
    base.mkdir(exist_ok=True)
    tar_dir = base / '_tars'
    tar_dir.mkdir(exist_ok=True)
    url = 'http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar'
    tar_path = tar_dir / url.rsplit('/', 1)[-1]
    if not tar_path.exists():
        print('Downloading', tar_path.name)
        urllib.request.urlretrieve(url, tar_path)
    if not (base / 'VOC2007').exists():
        print('Extracting', tar_path.name)
        with tarfile.open(tar_path) as tar:
            tar.extractall(base)
    return find_voc_root()

VOC_ROOT = find_voc_root()
if VOC_ROOT is None:
    print('VOC 2007 test was not found locally. Downloading it now...')
    VOC_ROOT = download_voc2007_test()

if VOC_ROOT is None:
    raise FileNotFoundError('Could not locate or download VOC 2007 test split.')

print('VOC_ROOT =', VOC_ROOT)


VOC 2007 test was not found locally. Downloading it now...
Extracting VOCtest_06-Nov-2007.tar
VOC_ROOT = /home/hm/4th_sem/study/cs106/treerl/VOCdevkit/VOCdevkit


In [4]:
# ============================================================
# Cell 4: Tree-RL definitions from tree_rl.ipynb
# ============================================================
ACTION_NAMES = [
    'scale_top_left', 'scale_top_right', 'scale_bottom_left',
    'scale_bottom_right', 'scale_center',
    'trans_left', 'trans_right', 'trans_up', 'trans_down',
    'trans_wider', 'trans_narrower', 'trans_taller', 'trans_shorter',
]

def apply_action(window, action_id, sr=0.55, tr=0.25, img_w=1, img_h=1):
    x1, y1, x2, y2 = window
    w, h = x2 - x1, y2 - y1
    if action_id < 5:
        sw, sh = w * sr, h * sr
        offsets = [(0, 0), (w - sw, 0), (0, h - sh), (w - sw, h - sh), ((w - sw) / 2, (h - sh) / 2)]
        dx, dy = offsets[action_id]
        nx1, ny1, nx2, ny2 = x1 + dx, y1 + dy, x1 + dx + sw, y1 + dy + sh
    else:
        dx, dy = w * tr, h * tr
        aid = action_id - 5
        if aid == 0:
            nx1, ny1, nx2, ny2 = x1 - dx, y1, x2 - dx, y2
        elif aid == 1:
            nx1, ny1, nx2, ny2 = x1 + dx, y1, x2 + dx, y2
        elif aid == 2:
            nx1, ny1, nx2, ny2 = x1, y1 - dy, x2, y2 - dy
        elif aid == 3:
            nx1, ny1, nx2, ny2 = x1, y1 + dy, x2, y2 + dy
        elif aid == 4:
            nx1, ny1, nx2, ny2 = x1 - dx, y1, x2 + dx, y2
        elif aid == 5:
            nx1, ny1, nx2, ny2 = x1 + dx, y1, x2 - dx, y2
        elif aid == 6:
            nx1, ny1, nx2, ny2 = x1, y1 - dy, x2, y2 + dy
        else:
            nx1, ny1, nx2, ny2 = x1, y1 + dy, x2, y2 - dy
    nx1 = max(0, min(nx1, img_w - 2))
    ny1 = max(0, min(ny1, img_h - 2))
    nx2 = max(nx1 + 1, min(nx2, img_w))
    ny2 = max(ny1 + 1, min(ny2, img_h))
    return np.array([nx1, ny1, nx2, ny2], dtype=np.float32)

def encode_history(history, hlen=50, nact=13):
    enc = np.zeros(hlen * nact, dtype=np.float32)
    for i, action in enumerate(history[-hlen:]):
        enc[i * nact + action] = 1.0
    return enc

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def compute_reward(cur, nxt, gts, hit_flags, thr=0.5, bonus=5.0):
    new_flags = hit_flags.copy()
    for i, gt in enumerate(gts):
        if hit_flags[i] < 1 and iou(nxt, gt) >= thr:
            new_flags[i] = 1
            return bonus, new_flags
    max_sign = -1.0
    for gt in gts:
        if iou(nxt, gt) - iou(cur, gt) > 0:
            max_sign = 1.0
            break
    return max_sign, new_flags

def recall_curve(proposals_list, gt_list, iou_thrs=IOU_THRESHOLDS, size_thr=SIZE_THRESHOLD):
    res = {'all': {}, 'large': {}, 'small': {}}
    for thr in iou_thrs:
        counts = {'all': [0, 0], 'large': [0, 0], 'small': [0, 0]}
        for props, gts in zip(proposals_list, gt_list):
            for gt in gts:
                area = (gt[2] - gt[0]) * (gt[3] - gt[1])
                sk = 'large' if area > size_thr else 'small'
                hit = int(any(iou(p, gt) >= thr for p in props))
                counts['all'][0] += hit
                counts['all'][1] += 1
                counts[sk][0] += hit
                counts[sk][1] += 1
        for k in res:
            total = counts[k][1]
            res[k][thr] = counts[k][0] / total if total else 0.0
    return res

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]
CLS2IDX = {c: i for i, c in enumerate(VOC_CLASSES)}

class VOCDataset(Dataset):
    def __init__(self, root, years=['2007'], split='test'):
        self.root = Path(root)
        self.split = split
        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.samples = []
        for year in years:
            sf = self.root / f'VOC{year}' / 'ImageSets' / 'Main' / f'{split}.txt'
            with open(sf) as f:
                self.samples += [(year, line.strip()) for line in f if line.strip()]
        print(f'[VOCDataset] {split}: {len(self.samples)} images from {years}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        year, iid = self.samples[idx]
        img_path = self.root / f'VOC{year}' / 'JPEGImages' / f'{iid}.jpg'
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        boxes, labels = [], []
        ann = self.root / f'VOC{year}' / 'Annotations' / f'{iid}.xml'
        for obj in ET.parse(ann).getroot().findall('object'):
            diff = obj.find('difficult')
            if diff is not None and int(diff.text) == 1:
                continue
            cls = obj.find('name').text.strip().lower()
            if cls not in CLS2IDX:
                continue
            bb = obj.find('bndbox')
            x1 = max(0, float(bb.find('xmin').text) - 1)
            y1 = max(0, float(bb.find('ymin').text) - 1)
            x2 = min(W, float(bb.find('xmax').text) - 1)
            y2 = min(H, float(bb.find('ymax').text) - 1)
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(CLS2IDX[cls])
        gt = np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)
        return {'image': self.transform(img), 'img_id': iid, 'year': year, 'img_w': W, 'img_h': H, 'gt_boxes': gt}

class VGG16Extractor(nn.Module):
    def __init__(self, roi_size=7):
        super().__init__()
        self.roi_size = roi_size
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.conv = nn.Sequential(*list(vgg.features.children())[:30])
        self.fc6 = nn.Sequential(nn.Linear(512 * roi_size * roi_size, 4096), nn.ReLU(inplace=True), nn.Dropout(0.5))
        if roi_size == 7:
            self.fc6[0].weight.data = vgg.classifier[0].weight.data
            self.fc6[0].bias.data = vgg.classifier[0].bias.data
        for p in self.conv.parameters():
            p.requires_grad = False

    def get_fmap(self, img):
        with torch.no_grad():
            return self.conv(img)

    def roi_feat(self, fmap, boxes, scale=1 / 16):
        bidx = torch.zeros(len(boxes), 1, dtype=boxes.dtype, device=boxes.device)
        rois = torch.cat([bidx, boxes], 1)
        pooled = ops.roi_pool(fmap, rois, (self.roi_size, self.roi_size), scale)
        return self.fc6(pooled.view(len(boxes), -1))

class QNetwork(nn.Module):
    def __init__(self, state_dim=8842, hidden=2048, n_act=13):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_act),
        )

    def forward(self, x):
        return self.net(x)

    def two_best(self, s):
        with torch.no_grad():
            q = self.forward(s.unsqueeze(0)).squeeze(0)
            return int(q[:5].argmax()), int(q[5:].argmax()) + 5

    def greedy(self, s):
        with torch.no_grad():
            return int(self.forward(s.unsqueeze(0)).squeeze(0).argmax())

class TreeRLAgent:
    def __init__(self, extractor, q_net, target_net, cfg, device):
        self.ext = extractor
        self.q = q_net
        self.tgt = target_net
        self.dev = device
        self.cfg = cfg
        self.roi_scale = 1 / 16

    def _whole_feat_np(self, fmap, W, H):
        box = torch.tensor([[0, 0, W, H]], dtype=torch.float32, device=self.dev)
        with torch.inference_mode():
            return self.ext.roi_feat(fmap, box, self.roi_scale).squeeze(0).cpu().numpy()

    def _window_feat_np(self, fmap, box):
        wb = torch.from_numpy(np.asarray([box], dtype=np.float32)).to(self.dev)
        with torch.inference_mode():
            return self.ext.roi_feat(fmap, wb, self.roi_scale).squeeze(0).cpu().numpy()

    def _build_state_np(self, gf_np, rf_np, hist):
        return np.concatenate([
            rf_np, gf_np, encode_history(hist, self.cfg['history_len'], self.cfg['num_actions'])
        ]).astype(np.float32, copy=False)

    def tree_search(self, fmap, W, H, levels=5):
        gf_np = self._whole_feat_np(fmap, W, H)
        props = []
        queue = deque()
        queue.append((np.array([0, 0, W, H], dtype=np.float32), [], 1))
        while queue:
            win, hist, lvl = queue.popleft()
            props.append(win.copy())
            if lvl >= levels:
                continue
            rf_np = self._window_feat_np(fmap, win)
            state = self._build_state_np(gf_np, rf_np, hist)
            st = torch.from_numpy(state).to(self.dev)
            bs, bt = self.q.two_best(st)
            for action in [bs, bt]:
                nw = apply_action(win, action, self.cfg['scaling_ratio'], self.cfg['trans_ratio'], W, H)
                queue.append((nw, hist + [action], lvl + 1))
        return props


In [5]:
# ============================================================
# Cell 5: Checkpoints, feature cache, and helpers
# ============================================================
def checkpoint_paths(model_dir=MODEL_DIR):
    paths = sorted(Path(model_dir).glob('epoch_*.pth'))
    expected = {f'epoch_{i:03d}.pth' for i in range(1, 26)}
    found = {p.name for p in paths}
    missing = sorted(expected - found)
    if missing:
        raise FileNotFoundError(f'Missing checkpoint(s): {missing}')
    return [Path(model_dir) / f'epoch_{i:03d}.pth' for i in range(1, 26)]

def infer_q_dims(q_state):
    first = q_state['net.0.weight']
    last = q_state['net.6.weight']
    return int(first.shape[1]), int(first.shape[0]), int(last.shape[0])

def cfg_from_checkpoint(ckpt):
    cfg = dict(ckpt.get('cfg') or {})
    q_state = ckpt['q_net']
    state_dim, hidden, n_act = infer_q_dims(q_state)
    cfg.setdefault('roi_pool_size', 7)
    cfg['state_dim'] = state_dim
    cfg['hidden_dim'] = hidden
    cfg['num_actions'] = n_act
    cfg.setdefault('history_len', (state_dim - 8192) // n_act)
    cfg.setdefault('scaling_ratio', 0.55)
    cfg.setdefault('trans_ratio', 0.25)
    cfg.setdefault('iou_threshold', 0.5)
    cfg.setdefault('first_hit_bonus', 5.0)
    cfg.setdefault('max_steps', 50)
    cfg.setdefault('gamma', 0.9)
    cfg.setdefault('double_dqn', True)
    return cfg

def load_model_pair(path, device=DEVICE):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    cfg = cfg_from_checkpoint(ckpt)
    q_net = QNetwork(cfg['state_dim'], cfg['hidden_dim'], cfg['num_actions']).to(device)
    target_net = QNetwork(cfg['state_dim'], cfg['hidden_dim'], cfg['num_actions']).to(device)
    q_net.load_state_dict(ckpt['q_net'])
    target_net.load_state_dict(ckpt.get('target_net', ckpt['q_net']))
    q_net.eval()
    target_net.eval()
    for p in target_net.parameters():
        p.requires_grad = False
    return ckpt, cfg, q_net, target_net

def load_or_extract_fmap(extractor, sample, split='test'):
    cache_dir = FEATURE_CACHE_DIR / f'VOC{sample["year"]}' / split
    cache_dir.mkdir(parents=True, exist_ok=True)
    out = cache_dir / f'{sample["img_id"]}.pt'
    if out.exists():
        d = torch.load(out, map_location='cpu', weights_only=False)
        fmap = d['feature_map'].float().unsqueeze(0).to(DEVICE, non_blocking=True)
        return fmap, int(d['img_w']), int(d['img_h'])

    img_t = sample['image'].unsqueeze(0).to(DEVICE, non_blocking=True)
    with torch.inference_mode():
        fmap = extractor.get_fmap(img_t).squeeze(0).contiguous().half().cpu()
    torch.save({'feature_map': fmap, 'img_w': sample['img_w'], 'img_h': sample['img_h']}, out)
    return fmap.float().unsqueeze(0).to(DEVICE, non_blocking=True), sample['img_w'], sample['img_h']

paths = checkpoint_paths()
first_ckpt = torch.load(paths[0], map_location='cpu', weights_only=False)
first_cfg = cfg_from_checkpoint(first_ckpt)
print('Found checkpoints:', len(paths))
print('Architecture:', {k: first_cfg[k] for k in ['state_dim', 'hidden_dim', 'num_actions', 'history_len', 'roi_pool_size']})


Found checkpoints: 25
Architecture: {'state_dim': 8842, 'hidden_dim': 2048, 'num_actions': 13, 'history_len': 50, 'roi_pool_size': 7}


In [6]:
# ============================================================
# Cell 6: Run evaluation over all epochs
# ============================================================
test_ds = VOCDataset(VOC_ROOT, years=['2007'], split='test')
n_images = len(test_ds) if MAX_IMAGES is None else min(MAX_IMAGES, len(test_ds))
indices = list(range(n_images))

extractor = VGG16Extractor(first_cfg['roi_pool_size']).to(DEVICE).eval()

rows = []
for path in tqdm(paths, desc='Checkpoints'):
    ckpt, cfg, q_net, target_net = load_model_pair(path)
    agent = TreeRLAgent(extractor, q_net, target_net, cfg, DEVICE)

    all_props = []
    all_gts = []
    for idx in tqdm(indices, desc=path.stem, leave=False):
        sample = test_ds[idx]
        if len(sample['gt_boxes']) == 0:
            continue
        fmap, W, H = load_or_extract_fmap(extractor, sample, split='test')
        with torch.inference_mode():
            props = agent.tree_search(fmap, W, H, levels=LEVELS)
        all_props.append(props)
        all_gts.append(sample['gt_boxes'])

    recall = recall_curve(all_props, all_gts)
    row = {
        'checkpoint': path.name,
        'epoch': int(ckpt.get('epoch', -1)) + 1,
        'global_step': int(ckpt.get('global_step', -1)),
        'levels': LEVELS,
        'proposals': 2 ** LEVELS - 1,
        'images_evaluated': len(all_gts),
    }
    for group in ['all', 'large', 'small']:
        for thr in IOU_THRESHOLDS:
            row[f'recall_{group}_{thr:.1f}'] = recall[group][thr] * 100
    rows.append(row)

    print(
        f"{path.name}: R@0.5={row['recall_all_0.5']:.2f}, "
        f"R@0.7={row['recall_all_0.7']:.2f}"
    )

fieldnames = list(rows[0].keys()) if rows else []
with open(RESULTS_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print('Saved:', RESULTS_CSV)


[VOCDataset] test: 4952 images from ['2007']


Checkpoints:   0%|          | 0/25 [00:00<?, ?it/s]

epoch_001:   0%|          | 0/4952 [00:00<?, ?it/s]

epoch_001.pth: R@0.5=40.08, R@0.7=16.59


epoch_002:   0%|          | 0/4952 [00:00<?, ?it/s]

...

epoch_023.pth: R@0.5=42.39, R@0.7=18.97


epoch_024:   0%|          | 0/4952 [00:00<?, ?it/s]

epoch_024.pth: R@0.5=42.66, R@0.7=18.49


epoch_025:   0%|          | 0/4952 [00:00<?, ?it/s]

epoch_025.pth: R@0.5=42.29, R@0.7=18.84
Saved: /home/hm/4th_sem/study/cs106/treerl/tree_rl_all_epochs_eval.csv


In [7]:
# ============================================================
# Cell 7: Display and pick best models
# ============================================================
try:
    import pandas as pd
    df = pd.read_csv(RESULTS_CSV)
    display(df)
    print('Best by recall_all_0.5')
    display(df.sort_values('recall_all_0.5', ascending=False).head(5))
except Exception as exc:
    print('Pandas display skipped:', type(exc).__name__, exc)
    with open(RESULTS_CSV) as f:
        print(f.read())


,checkpoint,epoch,global_step,levels,proposals,images_evaluated,recall_all_0.5,recall_all_0.6,recall_all_0.7,recall_all_0.8,recall_large_0.5,recall_large_0.6,recall_large_0.7,recall_large_0.8,recall_small_0.5,recall_small_0.6,recall_small_0.7,recall_small_0.8
0,epoch_001.pth,1,8076,5,31,4952,40.076463,28.548870,16.589096,7.239029,45.122294,32.190048,18.704901,8.162309,0.514328,0.000000,0.000000,0.0
1,epoch_002.pth,2,16351,5,31,4952,39.835439,28.631981,16.481051,7.081117,44.803674,32.246275,18.573704,7.984256,0.881705,0.293902,0.073475,0.0
2,epoch_003.pth,3,24626,5,31,4952,41.597407,30.460439,17.960439,7.438497,46.827851,34.326680,20.251148,8.387218,0.587803,0.146951,0.000000,0.0
3,epoch_004.pth,4,32901,5,31,4952,41.722074,30.335771,17.727726,7.471742,46.987161,34.176741,19.988755,8.424702,0.440852,0.220426,0.000000,0.0
4,epoch_005.pth,5,41176,5,31,4952,41.406250,30.194481,17.386968,7.255652,46.584200,34.008059,19.604536,8.181051,0.808229,0.293902,0.000000,0.0
5,epoch_006.pth,6,49451,5,31,4952,41.738697,30.526928,18.234707,7.912234,46.977790,34.401649,20.560397,8.921376,0.661278,0.146951,0.000000,0.0
6,epoch_007.pth,7,57726,5,31,4952,42.677859,31.607380,18.932846,8.244681,47.971137,35.582420,21.338206,9.296223,1.175606,0.440852,0.073475,0.0
7,epoch_008.pth,8,66001,5,31,4952,42.029588,30.626662,18.592088,7.978723,47.268297,34.495361,20.953987,8.996345,0.955180,0.293902,0.073475,0.0
8,epoch_009.pth,9,74276,5,31,4952,41.813497,30.601729,18.251330,7.920545,47.024646,34.485990,20.579140,8.930747,0.955180,0.146951,0.000000,0.0
9,epoch_010.pth,10,82551,5,31,4952,42.694481,31.732048,19.007646,8.361037,48.055477,35.760472,21.431918,9.427420,0.661278,0.146951,0.000000,0.0


Best by recall_all_0.5


,checkpoint,epoch,global_step,levels,proposals,images_evaluated,recall_all_0.5,recall_all_0.6,recall_all_0.7,recall_all_0.8,recall_large_0.5,recall_large_0.6,recall_large_0.7,recall_large_0.8,recall_small_0.5,recall_small_0.6,recall_small_0.7,recall_small_0.8
17,epoch_018.pth,18,148751,5,31,4952,43.276263,31.898271,19.215426,8.660239,48.711461,35.947896,21.666198,9.764783,0.661278,0.146951,0.000000,0.0
12,epoch_013.pth,13,107376,5,31,4952,43.018617,32.089428,19.581117,8.859707,48.392840,36.163434,22.078531,9.989692,0.881705,0.146951,0.000000,0.0
18,epoch_019.pth,19,157026,5,31,4952,42.943816,31.590758,18.858045,8.352726,48.299128,35.591791,21.263237,9.418049,0.955180,0.220426,0.000000,0.0
19,epoch_020.pth,20,165301,5,31,4952,42.927194,31.499335,18.791556,8.552194,48.271015,35.479337,21.188267,9.642958,1.028655,0.293902,0.000000,0.0
13,epoch_014.pth,14,115651,5,31,4952,42.827460,31.216755,19.232048,8.626995,48.214788,35.179458,21.675569,9.727298,0.587803,0.146951,0.073475,0.0
